# So sánh trực tiếp UCB-009 với Epsilon-Greedy trên 20 seed

Notebook dùng chung cho hai máy Kaggle và mỗi máy chỉ chạy một mô hình độc lập.

- Máy 1: MODEL_TO_RUN = "UNCERTAINTY_AWARE_UCB_009".
- Máy 2: MODEL_TO_RUN = "EPSILON_GREEDY".
- Dùng đúng 20 seed 41–60, tương ứng range(41, 61). Nếu chạy cả seed 61 sẽ thành 21 seed.
- Hai mô hình dùng cùng HPG BAD train/test, Q-network, 45 episode, transaction fee, reward shaping, frozen scaler chỉ fit trên train và Huber TD loss.
- UCB giữ đúng VAE vae_019 và UCB ucb_009, gồm Cost Network và confidence-weighted risk penalty.
- Epsilon-Greedy giữ lịch epsilon của Battle: 1.0, decay 0.95, minimum 0.05; không dùng VAE/Cost Network.
- Mỗi máy checkpoint và xuất báo cáo riêng. Cell cuối có chế độ tùy chọn để gộp hai CSV sau khi tải kết quả về cùng một notebook.

Đánh giá test theo từng episode chỉ dùng để chẩn đoán overfit; không early-stop và không cập nhật mô hình bằng dữ liệu test.

In [ ]:
!if [ ! -d SARSA_FinancialRL ]; then git clone https://github.com/kohi-vip/SARSA_FinancialRL.git; else echo 'SARSA_FinancialRL already exists'; fi


In [ ]:
!pip install numpy pandas matplotlib tqdm torch TA-Lib optuna tabulate


In [ ]:
# CODE 1 — Chọn một trong hai mô hình và khóa cấu hình so sánh
from __future__ import annotations
import gc, json, os, random, time, traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from tqdm.auto import tqdm

# Máy 1 dùng UNCERTAINTY_AWARE_UCB_009; máy 2 dùng EPSILON_GREEDY.
MODEL_TO_RUN = "UNCERTAINTY_AWARE_UCB_009"
RUN_TRAINING = True
RESUME = True
SEEDS = tuple(range(41, 61))  # 20 seed: 41..60.

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GAMMA, NN_LR, COST_LR, EPISODES = 0.95, 5e-5, 1e-3, 45
SARSA_ALPHA, Q_WEIGHT_DECAY, BATCH_SIZE = 0.60, 1e-4, 128
LATENT_DIM, BALANCE_INIT, TRANSACTION_FEE = 16, 1_000.0, 0.001
W_RISK, W_STABILITY, ZETA = 0.15, 0.05, 0.05
ACTION_VALUES = np.arange(-5, 6, dtype=np.int64)
RISK_FREE_RATE_PERCENT, SCALER_SEED = 2.0, 43
CURRENT_RUN_SEED = SEEDS[0]

LOCKED_VAE_019 = {
    "vae_latent_dim": 16, "vae_lr": 1e-4, "vae_beta_kl": 5e-2,
    "vae_batch_size": 256, "bootstrap_trajectories": 5,
    "bootstrap_updates": 100, "online_aux_updates": 1,
    "vae_replay_capacity": 50_000,
}
MODEL_CONFIGS = {
    "UNCERTAINTY_AWARE_UCB_009": {
        "model_key": "UNCERTAINTY_AWARE_UCB_009",
        "label": "Uncertainty-Aware UCB-VAE — vae_019 + ucb_009",
        "strategy": "ucb",
        "use_cost": True, "robust_loss": True,
        "weight_decay": Q_WEIGHT_DECAY, "reward_shaping": True,
        "kl_reduction": "sum",
        "beta_0": 0.03, "beta_decay": 0.90, "beta_min": 0.01,
        **LOCKED_VAE_019,
    },
    "EPSILON_GREEDY": {
        "model_key": "EPSILON_GREEDY",
        "label": "Epsilon-Greedy — Battle strategy",
        "strategy": "epsilon",
        "use_cost": False, "robust_loss": True,
        "weight_decay": 0.0, "reward_shaping": True,
        "epsilon_init": 1.0, "epsilon_decay": 0.95, "epsilon_min": 0.05,
    },
}
if MODEL_TO_RUN not in MODEL_CONFIGS:
    raise ValueError(f"MODEL_TO_RUN phải thuộc {list(MODEL_CONFIGS)}.")
SELECTED_CONFIG = dict(MODEL_CONFIGS[MODEL_TO_RUN])
MODEL_TAG = MODEL_TO_RUN.lower()

def set_seed(seed: int) -> None:
    global CURRENT_RUN_SEED
    CURRENT_RUN_SEED = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try: torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError: torch.use_deterministic_algorithms(True)

set_seed(SEEDS[0])
print({"device": str(DEVICE), "model": MODEL_TO_RUN, "config": SELECTED_CONFIG,
       "seeds": SEEDS, "runs": len(SEEDS)})

In [ ]:
# CODE 2 — Dữ liệu, môi trường cải tiến và frozen scaler chung cho hai chiến lược
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "SARSA_FinancialRL", Path("/kaggle/working/SARSA_FinancialRL"), *cwd.parents]
    for candidate in candidates:
        if (candidate / "data" / "data_storer" / "data_research").exists(): return candidate
    raise FileNotFoundError("Không tìm thấy project root chứa data/data_storer/data_research.")

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data" / "data_storer" / "data_research"
TRAIN_CSV = DATA_ROOT / "train" / "bad_train_HPG.csv"
TEST_CSV = DATA_ROOT / "test" / "bad_test_HPG.csv"
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else PROJECT_ROOT / "kaggle_working"
OUTPUT_DIR = OUTPUT_ROOT / "UCB009_vs_EpsilonGreedy_20seed" / MODEL_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REQUIRED_COLUMNS = ["time", "close", "MACD", "RSI", "CCI", "ADX"]

def load_hpg_bad() -> Tuple[pd.DataFrame, pd.DataFrame]:
    train, test = pd.read_csv(TRAIN_CSV), pd.read_csv(TEST_CSV)
    for label, frame in (("train", train), ("test", test)):
        missing = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
        if missing: raise ValueError(f"HPG BAD {label} thiếu cột: {missing}")
        frame["time"] = pd.to_datetime(frame["time"], errors="raise")
        frame[REQUIRED_COLUMNS[1:]] = frame[REQUIRED_COLUMNS[1:]].apply(pd.to_numeric, errors="coerce")
        if frame[REQUIRED_COLUMNS[1:]].isna().any().any():
            raise ValueError(f"HPG BAD {label} chứa NaN hoặc giá trị không hợp lệ.")
    if train["time"].max() >= test["time"].min(): raise ValueError("Train/Test chồng lấn thời gian.")
    return train.reset_index(drop=True), test.reset_index(drop=True)

def raw_state(row: pd.Series, cash: float, position: int) -> np.ndarray:
    return np.asarray([row["close"], cash, position, row["MACD"], row["RSI"], row["CCI"], row["ADX"]], dtype=np.float32)

class TradingEnv:
    """Môi trường và reward cải tiến giữ nguyên từ tim_tham_so.ipynb."""
    def __init__(self, data: pd.DataFrame, reward_shaping: bool, initial_cash: float = BALANCE_INIT):
        if len(data) < 2: raise ValueError("TradingEnv cần ít nhất hai quan sát.")
        self.data = data.reset_index(drop=True); self.reward_shaping = bool(reward_shaping)
        self.initial_cash = float(initial_cash); self.reset()
    def reset(self) -> np.ndarray:
        self.index, self.cash, self.position = 0, self.initial_cash, 0
        self.peak_value = self.initial_cash; self.portfolio_history = [self.initial_cash]; self.var_targets = []
        return self._state()
    def _state(self) -> np.ndarray:
        return raw_state(self.data.iloc[self.index], self.cash, self.position)
    def _execute(self, requested: int, price: float) -> int:
        if requested > 0:
            executed = min(int(requested), int(self.cash // (price * (1.0 + TRANSACTION_FEE))))
        else:
            executed = -min(-int(requested), self.position)
        traded_value = abs(executed) * price
        self.cash -= executed * price + traded_value * TRANSACTION_FEE
        self.position += executed
        return executed
    def step(self, action: int):
        current_price = float(self.data.iloc[self.index]["close"])
        previous_value = self.cash + self.position * current_price
        executed = self._execute(int(action), current_price)
        self.index += 1
        next_price = float(self.data.iloc[self.index]["close"])
        portfolio_value = self.cash + self.position * next_price
        raw_profit = portfolio_value - previous_value
        portfolio_return = raw_profit / max(abs(previous_value), 1e-8)
        var_target = max(-portfolio_return, 0.0)
        self.peak_value = max(self.peak_value, portfolio_value)
        drawdown = (self.peak_value - portfolio_value) / max(self.peak_value, 1e-8)
        asset_return = next_price / max(current_price, 1e-8) - 1.0
        reward = raw_profit
        if self.reward_shaping: reward -= W_RISK * abs(drawdown) + W_STABILITY * abs(asset_return)
        self.portfolio_history.append(float(portfolio_value)); self.var_targets.append(float(var_target))
        done = self.index >= len(self.data) - 1
        info = {"raw_profit": float(raw_profit), "portfolio_return": float(portfolio_return),
                "var_target": float(var_target), "drawdown": float(drawdown),
                "executed_action": int(executed), "portfolio_value": float(portfolio_value)}
        return self._state(), float(reward), done, info

@dataclass(frozen=True)
class FrozenScaler:
    mean: np.ndarray
    std: np.ndarray
    def transform(self, states: np.ndarray) -> np.ndarray:
        return (np.asarray(states, dtype=np.float32) - self.mean) / self.std

def calibration_states(train: pd.DataFrame, trajectories: int = 5) -> np.ndarray:
    rng = np.random.default_rng(SCALER_SEED); states = []
    for _ in range(trajectories):
        env = TradingEnv(train, reward_shaping=False); state, done = env.reset(), False
        while not done:
            states.append(state.copy()); state, _, done, _ = env.step(int(rng.choice(ACTION_VALUES)))
    return np.asarray(states, dtype=np.float32)

train_hpg, test_hpg = load_hpg_bad()
calibration = calibration_states(train_hpg)
frozen_mean = calibration.mean(axis=0, dtype=np.float64).astype(np.float32)
frozen_std = np.maximum(calibration.std(axis=0, dtype=np.float64), 1e-6).astype(np.float32)
FROZEN_SCALER = FrozenScaler(frozen_mean.copy(), frozen_std.copy())
FROZEN_SCALER.mean.setflags(write=False); FROZEN_SCALER.std.setflags(write=False)
print({"train": (str(train_hpg.time.min().date()), str(train_hpg.time.max().date()), len(train_hpg)),
       "test": (str(test_hpg.time.min().date()), str(test_hpg.time.max().date()), len(test_hpg)),
       "output_dir": str(OUTPUT_DIR), "scaler_fit": "train only"})

In [ ]:
# Ã” CODE 3 â€” Q-Network, EpistemicVAE vÃ  Cost Network
class QNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(7, 32), nn.ReLU(), nn.Linear(32, 11))

    def forward(self, states: torch.Tensor) -> torch.Tensor:
        return self.net(states)


class EpistemicVAE(nn.Module):
    """Joint VAE: 7 state + 11 action one-hot â†’ latent 16 â†’ reconstruction 18."""
    def __init__(self, latent_dim: int = LATENT_DIM):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(18, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.fc_mu = nn.Linear(32, latent_dim)
        self.fc_logvar = nn.Linear(32, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 18))

    def encode(self, x: torch.Tensor):
        hidden = self.encoder(x)
        return self.fc_mu(hidden), self.fc_logvar(hidden)

    def forward(self, states: torch.Tensor, actions_onehot: torch.Tensor):
        x = torch.cat([states, actions_onehot], dim=-1)
        mu, logvar = self.encode(x)
        std = torch.exp(0.5 * logvar)
        reconstructed = self.decoder(mu + torch.randn_like(std) * std)
        return reconstructed, mu, logvar

    def compute_u_ep(self, states, actions_onehot, kl_reduction: str = "sum"):
        """Novelty vá»›i KLD sum/mean tÃ¹y ablation vÃ  probe zâº=Î¼+1.96Ïƒ."""
        x = torch.cat([states, actions_onehot], dim=-1)
        mu, logvar = self.encode(x)
        kl_terms = -0.5 * (1.0 + logvar - mu.square() - logvar.exp())
        if kl_reduction == "sum":
            kl = torch.sum(kl_terms, dim=-1)
        elif kl_reduction == "mean":
            kl = torch.mean(kl_terms, dim=-1)
        else:
            raise ValueError("kl_reduction pháº£i lÃ  'sum' hoáº·c 'mean'.")
        std = torch.exp(0.5 * logvar)
        reconstructed_95 = self.decoder(mu + 1.96 * std)
        reconstruction_error = torch.norm(x - reconstructed_95, p=2, dim=-1)
        return kl + reconstruction_error


class CostNetwork(nn.Module):
    """Æ¯á»›c lÆ°á»£ng VaR khÃ´ng Ã¢m tá»« concat(s_scaled, action_onehot) âˆˆ R^18."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(18, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, states: torch.Tensor, actions_onehot: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([states, actions_onehot], dim=-1)).squeeze(-1)


def onehot(action_indices: torch.Tensor) -> torch.Tensor:
    return F.one_hot(action_indices.long(), num_classes=11).to(dtype=torch.float32)


print(QNetwork(), "\n", EpistemicVAE(), "\n", CostNetwork())


In [ ]:
# BLOCK 4 â€” Loss, replay vÃ  Confidence-Weighted Risk Penalty (tham sá»‘ hÃ³a cho grid)
def reconstruction_loss(prediction, target, robust: bool):
    return F.huber_loss(prediction, target) if robust else F.mse_loss(prediction, target)

def vae_loss(reconstructed, target, mu, logvar, beta_kl: float, robust: bool):
    recon = reconstruction_loss(reconstructed, target, robust)
    kl = -0.5 * torch.mean(1.0 + logvar - mu.square() - logvar.exp())
    return recon + float(beta_kl) * kl, recon, kl

def cost_loss(predicted_var, var_target):
    return F.huber_loss(predicted_var, var_target)

class ReplayBuffer:
    def __init__(self, capacity: int):
        self.capacity = int(capacity); self.states = []; self.actions = []; self.var_targets = []
    def add(self, states, action_indices, var_targets):
        for state, action, var_target in zip(states, action_indices, var_targets):
            self.states.append(np.asarray(state, dtype=np.float32)); self.actions.append(int(action)); self.var_targets.append(float(var_target))
        overflow = len(self.states) - self.capacity
        if overflow > 0:
            del self.states[:overflow]; del self.actions[:overflow]; del self.var_targets[:overflow]
    def sample(self, batch_size: int):
        size = min(int(batch_size), len(self.states))
        if size == 0: raise RuntimeError("KhÃ´ng thá»ƒ sample replay buffer rá»—ng.")
        indices = np.random.choice(len(self.states), size=size, replace=False)
        return (np.asarray([self.states[i] for i in indices]), np.asarray([self.actions[i] for i in indices]),
                np.asarray([self.var_targets[i] for i in indices], dtype=np.float32))
    def __len__(self): return len(self.states)

def action_scores(q_network, vae, cost_network, state, beta, config, collect_details=False):
    scaled = FROZEN_SCALER.transform(np.asarray(state).reshape(1, -1))
    state_tensor = torch.as_tensor(scaled, dtype=torch.float32, device=DEVICE)
    state_batch = state_tensor.repeat(11, 1); action_batch = torch.eye(11, dtype=torch.float32, device=DEVICE)
    q_network.eval(); vae.eval(); cost_network.eval()
    with torch.no_grad():
        q_values = q_network(state_tensor).squeeze(0)
        novelty_raw = vae.compute_u_ep(state_batch, action_batch, config["kl_reduction"])
        novelty_scaled = novelty_raw / (novelty_raw.max() + 1e-8)
        predicted_var = cost_network(state_batch, action_batch)
        confidence = 1.0 / (1.0 + novelty_raw.clamp_min(0.0))
        penalty = torch.where(predicted_var < ZETA, torch.zeros_like(predicted_var), confidence * predicted_var)
        scores = q_values - penalty + float(beta) * novelty_scaled
        action_index = int(torch.argmax(scores).item())
    details = None if not collect_details else {
        "novelty": novelty_raw.detach().cpu().numpy(), "predicted_var": predicted_var.detach().cpu().numpy(),
        "penalty": penalty.detach().cpu().numpy(),
    }
    return int(ACTION_VALUES[action_index]), action_index, details

def auxiliary_update(vae, vae_optimizer, cost_network, cost_optimizer, replay, config):
    raw_states, action_indices, var_targets = replay.sample(config["vae_batch_size"])
    states = torch.as_tensor(FROZEN_SCALER.transform(raw_states), dtype=torch.float32, device=DEVICE)
    indices = torch.as_tensor(action_indices, dtype=torch.long, device=DEVICE); actions = onehot(indices)
    reconstructed, mu, logvar = vae(states, actions); target = torch.cat([states, actions], dim=-1)
    loss_vae, recon, kl = vae_loss(reconstructed, target, mu, logvar, config["vae_beta_kl"], config["robust_loss"])
    vae_optimizer.zero_grad(set_to_none=True); loss_vae.backward(); torch.nn.utils.clip_grad_norm_(vae.parameters(), 5.0); vae_optimizer.step()
    targets = torch.as_tensor(var_targets, dtype=torch.float32, device=DEVICE)
    predicted = cost_network(states, actions); loss_c = cost_loss(predicted, targets)
    cost_optimizer.zero_grad(set_to_none=True); loss_c.backward(); torch.nn.utils.clip_grad_norm_(cost_network.parameters(), 5.0); cost_optimizer.step()
    return float(loss_vae.detach().cpu()), float(recon.detach().cpu()), float(kl.detach().cpu()), float(loss_c.detach().cpu())

def random_bootstrap(replay: ReplayBuffer, config):
    rng = np.random.default_rng(CURRENT_RUN_SEED)
    for _ in range(int(config["bootstrap_trajectories"])):
        env = TradingEnv(train_hpg, reward_shaping=False); state, done = env.reset(), False
        states, actions, var_targets = [], [], []
        while not done:
            action_index = int(rng.integers(0, 11)); states.append(state.copy()); actions.append(action_index)
            state, _, done, info = env.step(int(ACTION_VALUES[action_index])); var_targets.append(info["var_target"])
        replay.add(states, actions, var_targets)

def collect_episode(env, q_network, vae, cost_network, beta, config):
    state, done = env.reset(), False; states, next_states, rewards, actions, dones, var_targets = [], [], [], [], [], []
    while not done:
        action, action_index, _ = action_scores(q_network, vae, cost_network, state, beta, config)
        next_state, reward, done, info = env.step(action)
        states.append(state.copy()); next_states.append(next_state.copy()); rewards.append(reward)
        actions.append(action_index); dones.append(done); var_targets.append(info["var_target"]); state = next_state
    next_actions = actions[1:] + [actions[-1]]
    return states, next_states, rewards, actions, next_actions, dones, var_targets

def q_update(q_network, optimizer, trajectory, robust: bool):
    states, next_states, rewards, actions, next_actions, dones, _ = trajectory
    s = torch.as_tensor(FROZEN_SCALER.transform(states), dtype=torch.float32, device=DEVICE)
    sn = torch.as_tensor(FROZEN_SCALER.transform(next_states), dtype=torch.float32, device=DEVICE)
    r = torch.as_tensor(rewards, dtype=torch.float32, device=DEVICE)
    a = torch.as_tensor(actions, dtype=torch.long, device=DEVICE); an = torch.as_tensor(next_actions, dtype=torch.long, device=DEVICE)
    terminal = torch.as_tensor(dones, dtype=torch.float32, device=DEVICE); losses = []; q_network.train()
    for start in range(0, len(states), BATCH_SIZE):
        sl = slice(start, min(start + BATCH_SIZE, len(states)))
        current = q_network(s[sl]).gather(1, a[sl, None]).squeeze(1)
        with torch.no_grad():
            following = q_network(sn[sl]).gather(1, an[sl, None]).squeeze(1)
            td_target = r[sl] + GAMMA * (1.0 - terminal[sl]) * following
            target = (1.0 - SARSA_ALPHA) * current.detach() + SARSA_ALPHA * td_target
        loss = F.huber_loss(current, target) if robust else F.mse_loss(current, target)
        optimizer.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(q_network.parameters(), 5.0)
        optimizer.step(); losses.append(float(loss.detach().cpu()))
    return losses

In [ ]:
# CODE 5 — Huấn luyện độc lập một chiến lược trên 20 seed
def evaluation_frame(data: pd.DataFrame, previous_row: Optional[pd.Series] = None) -> pd.DataFrame:
    return data if previous_row is None else pd.concat([previous_row.to_frame().T, data], ignore_index=True)

def epsilon_action(q_network, state, epsilon: float, explore: bool):
    if explore and np.random.random() < float(epsilon):
        index = int(np.random.randint(0, len(ACTION_VALUES)))
    else:
        scaled = FROZEN_SCALER.transform(np.asarray(state).reshape(1, -1))
        tensor = torch.as_tensor(scaled, dtype=torch.float32, device=DEVICE)
        q_network.eval()
        with torch.no_grad(): index = int(torch.argmax(q_network(tensor).squeeze(0)).item())
    return int(ACTION_VALUES[index]), index

def collect_epsilon_episode(env, q_network, epsilon: float):
    state, done = env.reset(), False
    states, next_states, rewards, actions, dones, var_targets = [], [], [], [], [], []
    while not done:
        action, action_index = epsilon_action(q_network, state, epsilon, explore=True)
        next_state, reward, done, info = env.step(action)
        states.append(state.copy()); next_states.append(next_state.copy())
        rewards.append(reward); actions.append(action_index); dones.append(done)
        var_targets.append(info["var_target"]); state = next_state
    next_actions = actions[1:] + [actions[-1]]
    return states, next_states, rewards, actions, next_actions, dones, var_targets

def evaluate_strategy(q_network, vae, cost_network, control, config, data, previous_row=None):
    env = TradingEnv(evaluation_frame(data, previous_row), reward_shaping=config["reward_shaping"])
    state, done, novelty_values = env.reset(), False, []
    while not done:
        if config["strategy"] == "epsilon":
            action, _ = epsilon_action(q_network, state, epsilon=0.0, explore=False)
        else:
            action, _, details = action_scores(
                q_network, vae, cost_network, state, control, config, collect_details=True)
            novelty_values.extend(details["novelty"].tolist())
        state, _, done, _ = env.step(action)
    return (np.asarray(env.portfolio_history, dtype=np.float64),
            np.asarray(env.var_targets, dtype=np.float64),
            np.asarray(novelty_values, dtype=np.float64))

def period_metrics(portfolio: np.ndarray, dates: Sequence[pd.Timestamp], var_targets=None) -> Dict[str, float]:
    profit = float(portfolio[-1] - portfolio[0])
    roi = float(profit / max(abs(portfolio[0]), 1e-8) * 100.0)
    dates = pd.Series(dates).reset_index(drop=True)
    days = max((pd.Timestamp(dates.iloc[-1]) - pd.Timestamp(dates.iloc[0])).days, 1)
    ratio = float(portfolio[-1] / max(portfolio[0], 1e-8))
    arr = float((ratio ** (365.25 / days) - 1.0) * 100.0) if ratio > 0 else -100.0
    returns = np.diff(portfolio) / np.maximum(np.abs(portfolio[:-1]), 1e-8)
    annual_return = np.mean(returns) * 252.0 * 100.0 if len(returns) else 0.0
    volatility = np.std(returns) * np.sqrt(252.0) * 100.0 if len(returns) else 0.0
    sharpe = 0.0 if volatility < 1e-12 else float((annual_return - RISK_FREE_RATE_PERCENT) / volatility)
    peaks = np.maximum.accumulate(portfolio)
    mdd = float(abs(np.min((portfolio - peaks) / np.maximum(peaks, 1e-8))) * 100.0)
    violations = int(np.sum(np.asarray(var_targets) > ZETA)) if var_targets is not None else 0
    return {"profit": profit, "roi": roi, "arr": arr, "sharpe": sharpe,
            "max_drawdown": mdd, "violations": violations}

def run_one_seed(seed: int, progress_bar=None):
    set_seed(seed); config = dict(SELECTED_CONFIG)
    # Khởi tạo cùng thứ tự để Q-network có khởi tạo so sánh được giữa hai chiến lược.
    q_network = QNetwork().to(DEVICE)
    vae = EpistemicVAE(latent_dim=LATENT_DIM).to(DEVICE)
    cost_network = CostNetwork().to(DEVICE)
    q_optimizer = torch.optim.Adam(q_network.parameters(), lr=NN_LR, weight_decay=config["weight_decay"])
    vae_optimizer = (torch.optim.Adam(vae.parameters(), lr=config["vae_lr"])
                     if config["strategy"] == "ucb" else None)
    cost_optimizer = (torch.optim.Adam(cost_network.parameters(), lr=COST_LR)
                      if config.get("use_cost", False) else None)
    replay = ReplayBuffer(config["vae_replay_capacity"]) if config["strategy"] == "ucb" else None
    q_losses, vae_losses, recon_losses, kl_losses, cost_losses, curve_rows = [], [], [], [], [], []

    if config["strategy"] == "ucb":
        random_bootstrap(replay, config)
        if progress_bar is not None: progress_bar.set_postfix(seed=seed, phase="aux-warmup")
        for _ in range(int(config["bootstrap_updates"])):
            lv, lr, lk, lc = auxiliary_update(
                vae, vae_optimizer, cost_network, cost_optimizer, replay, config)
            vae_losses.append(lv); recon_losses.append(lr); kl_losses.append(lk); cost_losses.append(lc)

    for episode in range(EPISODES):
        if config["strategy"] == "epsilon":
            control = max(float(config["epsilon_min"]),
                          float(config["epsilon_init"]) * float(config["epsilon_decay"]) ** episode)
            trajectory = collect_epsilon_episode(
                TradingEnv(train_hpg, reward_shaping=config["reward_shaping"]),
                q_network, control)
        else:
            control = max(float(config["beta_min"]),
                          float(config["beta_0"]) * float(config["beta_decay"]) ** episode)
            trajectory = collect_episode(
                TradingEnv(train_hpg, reward_shaping=config["reward_shaping"]),
                q_network, vae, cost_network, control, config)
            replay.add(trajectory[0], trajectory[3], trajectory[6])
        q_losses.extend(q_update(q_network, q_optimizer, trajectory, config["robust_loss"]))
        if config["strategy"] == "ucb":
            for _ in range(int(config["online_aux_updates"])):
                lv, lr, lk, lc = auxiliary_update(
                    vae, vae_optimizer, cost_network, cost_optimizer, replay, config)
                vae_losses.append(lv); recon_losses.append(lr); kl_losses.append(lk); cost_losses.append(lc)

        train_portfolio, train_var, _ = evaluate_strategy(
            q_network, vae, cost_network, control, config, train_hpg)
        test_portfolio, test_var, _ = evaluate_strategy(
            q_network, vae, cost_network, control, config, test_hpg, train_hpg.iloc[-1])
        train_metrics = period_metrics(train_portfolio, train_hpg["time"], train_var)
        test_metrics = period_metrics(test_portfolio, test_hpg["time"], test_var)
        for split, values in (("train", train_metrics), ("test", test_metrics)):
            curve_rows.append({"model_key": MODEL_TO_RUN, "model": config["label"],
                               "seed": int(seed), "episode": episode + 1,
                               "split": split, "control": float(control), **values})
        if progress_bar is not None and ((episode + 1) % 5 == 0 or episode + 1 == EPISODES):
            progress_bar.set_postfix(seed=seed, episode=f"{episode + 1}/{EPISODES}",
                                     test_sharpe=f"{test_metrics['sharpe']:.3f}")

    train_portfolio, train_var, _ = evaluate_strategy(
        q_network, vae, cost_network, control, config, train_hpg)
    test_portfolio, test_var, novelty = evaluate_strategy(
        q_network, vae, cost_network, control, config, test_hpg, train_hpg.iloc[-1])
    train_metrics = period_metrics(train_portfolio, train_hpg["time"], train_var)
    test_metrics = period_metrics(test_portfolio, test_hpg["time"], test_var)
    metric_row = {
        "model_key": MODEL_TO_RUN, "model": config["label"], "strategy": config["strategy"],
        "seed": int(seed), "episodes": EPISODES, "gamma": GAMMA,
        "beta_0": config.get("beta_0", np.nan),
        "beta_decay": config.get("beta_decay", np.nan),
        "beta_min": config.get("beta_min", np.nan),
        "epsilon_init": config.get("epsilon_init", np.nan),
        "epsilon_decay": config.get("epsilon_decay", np.nan),
        "epsilon_min": config.get("epsilon_min", np.nan),
        "vae_latent_dim": config.get("vae_latent_dim", np.nan),
        "vae_lr": config.get("vae_lr", np.nan),
        "vae_beta_kl": config.get("vae_beta_kl", np.nan),
    }
    metric_row.update({f"train_{key}": value for key, value in train_metrics.items()})
    metric_row.update({f"test_{key}": value for key, value in test_metrics.items()})
    metric_row.update({
        "gap_profit": train_metrics["profit"] - test_metrics["profit"],
        "gap_roi": train_metrics["roi"] - test_metrics["roi"],
        "gap_arr": train_metrics["arr"] - test_metrics["arr"],
        "gap_sharpe": train_metrics["sharpe"] - test_metrics["sharpe"],
        "q_loss_mean": float(np.mean(q_losses)) if q_losses else np.nan,
        "vae_loss_mean": float(np.mean(vae_losses)) if vae_losses else np.nan,
        "cost_loss_mean": float(np.mean(cost_losses)) if cost_losses else np.nan,
        "novelty_mean": float(np.mean(novelty)) if len(novelty) else np.nan,
        "finite": bool(np.isfinite(np.asarray(
            list(train_metrics.values()) + list(test_metrics.values()), dtype=float)).all()),
    })
    loss_rows = []
    for loss_name, values in (("Q loss", q_losses), ("VAE loss", vae_losses), ("Cost loss", cost_losses)):
        for update, value in enumerate(values):
            loss_rows.append({"model_key": MODEL_TO_RUN, "model": config["label"],
                              "seed": int(seed), "loss_name": loss_name,
                              "update": update, "loss": float(value)})
    del q_network, vae, cost_network, q_optimizer, vae_optimizer, cost_optimizer, replay
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return metric_row, curve_rows, loss_rows

METRICS_CSV = OUTPUT_DIR / f"comparison_{MODEL_TAG}_20seed_metrics.csv"
CURVES_CSV = OUTPUT_DIR / f"comparison_{MODEL_TAG}_episode_curves.csv"
LOSSES_CSV = OUTPUT_DIR / f"comparison_{MODEL_TAG}_losses.csv"
ERRORS_CSV = OUTPUT_DIR / f"comparison_{MODEL_TAG}_errors.csv"

def append_frame(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, mode="a", header=not path.exists(), index=False)

def completed_seeds() -> set[int]:
    if not RESUME or not METRICS_CSV.exists(): return set()
    frame = pd.read_csv(METRICS_CSV)
    if set(frame["model_key"].astype(str)) != {MODEL_TO_RUN}:
        raise ValueError(f"CSV resume không khớp MODEL_TO_RUN={MODEL_TO_RUN}.")
    return set(frame["seed"].astype(int)).intersection(SEEDS)

done = completed_seeds()
if RUN_TRAINING:
    progress = tqdm(SEEDS, total=len(SEEDS), desc=f"{MODEL_TO_RUN} — 20 independent seeds",
                    unit="seed", dynamic_ncols=True, leave=True, mininterval=1.0)
    for seed in progress:
        if seed in done:
            progress.set_postfix(seed=seed, status="checkpoint-skip"); continue
        started = time.perf_counter()
        try:
            metric_row, curve_rows, loss_rows = run_one_seed(seed, progress_bar=progress)
            metric_row["elapsed_seconds"] = float(time.perf_counter() - started)
            append_frame(CURVES_CSV, pd.DataFrame(curve_rows))
            if loss_rows: append_frame(LOSSES_CSV, pd.DataFrame(loss_rows))
            append_frame(METRICS_CSV, pd.DataFrame([metric_row]))
            progress.set_postfix(seed=seed, test_profit=f"{metric_row['test_profit']:.2f}",
                                 test_sharpe=f"{metric_row['test_sharpe']:.3f}")
        except Exception as error:
            append_frame(ERRORS_CSV, pd.DataFrame([{"model_key": MODEL_TO_RUN, "seed": int(seed),
                "error_type": type(error).__name__, "error": str(error),
                "traceback": traceback.format_exc()}]))
            print(f"FAILED seed={seed}: {type(error).__name__}: {error}", flush=True)
        finally:
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    print("Đã hoàn tất vòng chạy. Metrics:", METRICS_CSV)
else:
    print("RUN_TRAINING=False — chỉ đọc checkpoint hiện có.")

In [ ]:
# CODE 6 — Báo cáo riêng cho mô hình trên máy hiện tại
if not METRICS_CSV.exists():
    raise RuntimeError(f"Chưa có kết quả: {METRICS_CSV}")

metrics = (pd.read_csv(METRICS_CSV)
           .drop_duplicates(["model_key", "seed"], keep="last")
           .sort_values("seed").reset_index(drop=True))
curves = (pd.read_csv(CURVES_CSV)
          .drop_duplicates(["model_key", "seed", "episode", "split"], keep="last")
          if CURVES_CSV.exists() else pd.DataFrame())
losses = (pd.read_csv(LOSSES_CSV)
          .drop_duplicates(["model_key", "seed", "loss_name", "update"], keep="last")
          if LOSSES_CSV.exists() else pd.DataFrame())
if set(metrics["model_key"].astype(str)) != {MODEL_TO_RUN}:
    raise ValueError("Metrics chứa mô hình khác với MODEL_TO_RUN.")

summary = {"model_key": MODEL_TO_RUN, "model": SELECTED_CONFIG["label"],
           "seeds_completed": int(metrics["seed"].nunique())}
for split in ("train", "test"):
    for metric in ("profit", "roi", "arr", "sharpe", "max_drawdown", "violations"):
        column = f"{split}_{metric}"
        summary[f"{column}_mean"] = float(metrics[column].mean())
        summary[f"{column}_std"] = float(metrics[column].std(ddof=1))
for metric in ("profit", "roi", "arr", "sharpe"):
    summary[f"gap_{metric}_mean"] = float(metrics[f"gap_{metric}"].mean())
    summary[f"gap_{metric}_std"] = float(metrics[f"gap_{metric}"].std(ddof=1))

summary_frame = pd.DataFrame([summary])
SUMMARY_CSV = OUTPUT_DIR / f"comparison_{MODEL_TAG}_20seed_summary.csv"
SUMMARY_JSON = OUTPUT_DIR / f"comparison_{MODEL_TAG}_20seed_report.json"
summary_frame.to_csv(SUMMARY_CSV, index=False)
SUMMARY_JSON.write_text(json.dumps({"selected_config": SELECTED_CONFIG,
    "seed_range": list(SEEDS), "summary": summary}, ensure_ascii=False, indent=2), encoding="utf-8")

columns = ["model", "seed", "train_roi", "test_roi", "gap_roi", "train_arr", "test_arr",
           "gap_arr", "train_sharpe", "test_sharpe", "gap_sharpe",
           "test_max_drawdown", "test_violations", "finite"]
try:
    print(metrics[columns].to_markdown(index=False, floatfmt=".6f"))
    print("\n### Mean ± Std"); print(summary_frame.to_markdown(index=False, floatfmt=".6f"))
except ImportError:
    print(metrics[columns].to_string(index=False)); print(summary_frame.to_string(index=False))

PLOT_PREFIX = OUTPUT_DIR / f"comparison_{MODEL_TAG}"

# Final test metrics mean ± std.
items = [("test_profit", "Final Profit"), ("test_roi", "ROI (%)"), ("test_arr", "ARR (%)"),
         ("test_sharpe", "Sharpe Ratio"), ("test_max_drawdown", "Max Drawdown (%)"),
         ("test_violations", "Constraint Violations")]
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, (column, title) in zip(axes.ravel(), items):
    std_value = metrics[column].std(ddof=1)
    std_value = float(std_value) if np.isfinite(std_value) else 0.0
    ax.bar([MODEL_TAG], [metrics[column].mean()], yerr=[std_value], capsize=5)
    ax.set_title(f"Test — {title} (mean ± std)"); ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
FINAL_METRICS_PNG = Path(f"{PLOT_PREFIX}_final_test_metrics_mean_std.png")
fig.savefig(FINAL_METRICS_PNG, dpi=220, bbox_inches="tight"); plt.show()

# Generalization gap theo episode.
GENERALIZATION_PNG = Path(f"{PLOT_PREFIX}_generalization_gap_by_episode.png")
if not curves.empty:
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    for ax, metric, title in zip(axes.ravel(), ("profit", "roi", "arr", "sharpe"),
                                 ("Profit", "ROI (%)", "ARR (%)", "Sharpe Ratio")):
        for split, style in (("train", "-"), ("test", "--")):
            grouped = curves[curves["split"] == split].groupby("episode")[metric].agg(["mean", "std"]).reset_index()
            x, y = grouped["episode"].to_numpy(), grouped["mean"].to_numpy()
            spread = grouped["std"].fillna(0.0).to_numpy()
            ax.plot(x, y, linestyle=style, label=f"{split} mean")
            ax.fill_between(x, y - spread, y + spread, alpha=0.14)
        ax.set(title=f"Generalization Gap — {title}", xlabel="Episode", ylabel=title)
        ax.grid(alpha=0.25); ax.legend()
    plt.tight_layout(); fig.savefig(GENERALIZATION_PNG, dpi=220, bbox_inches="tight"); plt.show()

# Phân phối test và gap train-test.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, column, title in zip(axes, ("test_roi", "test_arr", "test_sharpe"),
                             ("Test ROI (%)", "Test ARR (%)", "Test Sharpe")):
    ax.boxplot([metrics[column].dropna().to_numpy()], tick_labels=[MODEL_TAG], showmeans=True)
    ax.set_title(f"20-seed distribution — {title}"); ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
DISTRIBUTION_PNG = Path(f"{PLOT_PREFIX}_test_distributions.png")
fig.savefig(DISTRIBUTION_PNG, dpi=220, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, column, title in zip(axes, ("gap_roi", "gap_arr", "gap_sharpe"),
                             ("ROI gap", "ARR gap", "Sharpe gap")):
    ax.boxplot([metrics[column].dropna().to_numpy()], tick_labels=[MODEL_TAG], showmeans=True)
    ax.axhline(0.0, color="gray", linestyle="--", linewidth=1)
    ax.set_title(f"Train − Test — {title}"); ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
GAP_PNG = Path(f"{PLOT_PREFIX}_overfit_gap_distributions.png")
fig.savefig(GAP_PNG, dpi=220, bbox_inches="tight"); plt.show()

# Loss convergence.
LOSSES_PNG = Path(f"{PLOT_PREFIX}_loss_convergence.png")
if not losses.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, loss_name in zip(axes, ("Q loss", "VAE loss", "Cost loss")):
        grouped = losses[losses["loss_name"] == loss_name].groupby("update")["loss"].mean().reset_index()
        if not grouped.empty:
            ax.plot(grouped["update"], np.maximum(grouped["loss"], 1e-12))
            ax.set_yscale("log")
        ax.set(title=loss_name, xlabel="Update", ylabel="Loss"); ax.grid(alpha=0.25)
    plt.tight_layout(); fig.savefig(LOSSES_PNG, dpi=220, bbox_inches="tight"); plt.show()

print({"model": MODEL_TO_RUN, "seeds_completed": int(metrics.seed.nunique()),
       "missing_seeds": sorted(set(SEEDS) - set(metrics.seed.astype(int))),
       "warning": "Chỉ kết luận sau khi đủ 20 seed trên cả hai máy."})
print("\n=== FILE CỦA MÁY HIỆN TẠI ===")
for path in [METRICS_CSV, CURVES_CSV, LOSSES_CSV, SUMMARY_CSV, SUMMARY_JSON,
             FINAL_METRICS_PNG, GENERALIZATION_PNG, DISTRIBUTION_PNG, GAP_PNG, LOSSES_PNG]:
    if path.exists(): print("-", path)

In [ ]:
# CODE 7 — Tùy chọn gộp hai máy để so sánh trực tiếp sau khi cả hai hoàn tất
# Trên một notebook mới, attach output của hai máy vào /kaggle/input rồi đặt True.
COMPARE_TWO_MODELS = False

def discover_files(filename_pattern: str) -> List[Path]:
    roots = [OUTPUT_ROOT / "UCB009_vs_EpsilonGreedy_20seed"]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists(): roots.append(kaggle_input)
    files = []
    for root in roots:
        if root.exists(): files.extend(root.rglob(filename_pattern))
    return list(dict.fromkeys(path.resolve() for path in files))

if not COMPARE_TWO_MODELS:
    print("COMPARE_TWO_MODELS=False — bỏ qua gộp. Sau khi attach output hai máy, đổi thành True.")
else:
    expected_models = {"UNCERTAINTY_AWARE_UCB_009", "EPSILON_GREEDY"}
    metric_frames = []
    for path in discover_files("comparison_*_20seed_metrics.csv"):
        try:
            frame = pd.read_csv(path)
            if {"model_key", "seed", "test_profit", "test_sharpe"}.issubset(frame.columns):
                metric_frames.append(frame)
                print("Loaded metrics:", path)
        except (OSError, pd.errors.ParserError, pd.errors.EmptyDataError) as error:
            print("Bỏ qua", path, error)
    if not metric_frames:
        raise RuntimeError("Không tìm thấy metrics của hai máy.")
    combined = (pd.concat(metric_frames, ignore_index=True)
                .drop_duplicates(["model_key", "seed"], keep="last"))
    missing_models = expected_models - set(combined["model_key"].astype(str))
    if missing_models: raise RuntimeError(f"Thiếu kết quả của: {sorted(missing_models)}")
    seed_sets = {key: set(group["seed"].astype(int)) for key, group in combined.groupby("model_key")}
    expected_seed_set = set(SEEDS)
    invalid = {key: sorted(expected_seed_set.symmetric_difference(seed_sets.get(key, set())))
               for key in expected_models if seed_sets.get(key, set()) != expected_seed_set}
    if invalid:
        raise RuntimeError(f"Mỗi mô hình phải có đúng seed 41–60; chênh lệch: {invalid}")

    rows = []
    for model_key, group in combined.groupby("model_key"):
        row = {"model_key": model_key, "model": group["model"].iloc[0],
               "seeds": int(group.seed.nunique())}
        for metric in ("profit", "roi", "arr", "sharpe", "max_drawdown", "violations"):
            column = f"test_{metric}"
            row[f"{metric}_mean"] = float(group[column].mean())
            row[f"{metric}_std"] = float(group[column].std(ddof=1))
        for metric in ("roi", "arr", "sharpe"):
            row[f"gap_{metric}_mean"] = float(group[f"gap_{metric}"].mean())
        rows.append(row)
    comparison = pd.DataFrame(rows).sort_values("sharpe_mean", ascending=False)

    COMPARE_DIR = OUTPUT_ROOT / "UCB009_vs_EpsilonGreedy_20seed" / "combined_comparison"
    COMPARE_DIR.mkdir(parents=True, exist_ok=True)
    COMPARISON_CSV = COMPARE_DIR / "ucb009_vs_epsilon_20seed_summary.csv"
    PAIRED_CSV = COMPARE_DIR / "ucb009_vs_epsilon_paired_differences.csv"
    comparison.to_csv(COMPARISON_CSV, index=False)
    try: print(comparison.to_markdown(index=False, floatfmt=".6f"))
    except ImportError: print(comparison.to_string(index=False))

    # Chênh lệch ghép cặp: UCB - Epsilon trên đúng cùng seed.
    paired_rows = []
    for metric in ("profit", "roi", "arr", "sharpe", "max_drawdown", "violations"):
        pivot = combined.pivot(index="seed", columns="model_key", values=f"test_{metric}").dropna()
        difference = pivot["UNCERTAINTY_AWARE_UCB_009"] - pivot["EPSILON_GREEDY"]
        paired_rows.append({
            "metric": metric, "paired_seeds": len(difference),
            "ucb_minus_epsilon_mean": float(difference.mean()),
            "ucb_minus_epsilon_std": float(difference.std(ddof=1)),
            "ucb_wins": int((difference > 0).sum()) if metric not in ("max_drawdown", "violations") else int((difference < 0).sum()),
            "ties": int((difference == 0).sum()),
        })
    paired = pd.DataFrame(paired_rows)
    paired.to_csv(PAIRED_CSV, index=False)
    try: print("\n### Paired differences\n", paired.to_markdown(index=False, floatfmt=".6f"))
    except ImportError: print(paired.to_string(index=False))

    # Mean ± std trực tiếp.
    plot_items = [("profit", "Final Profit"), ("roi", "ROI (%)"), ("arr", "ARR (%)"),
                  ("sharpe", "Sharpe Ratio"), ("max_drawdown", "Max Drawdown (%)"),
                  ("violations", "Constraint Violations")]
    fig, axes = plt.subplots(2, 3, figsize=(18, 9))
    for ax, (metric, title) in zip(axes.ravel(), plot_items):
        ax.bar(comparison["model_key"], comparison[f"{metric}_mean"],
               yerr=comparison[f"{metric}_std"], capsize=5)
        ax.set_title(f"Test — {title}"); ax.grid(axis="y", alpha=0.25)
        ax.tick_params(axis="x", rotation=15)
    plt.tight_layout()
    COMPARISON_PNG = COMPARE_DIR / "ucb009_vs_epsilon_mean_std.png"
    fig.savefig(COMPARISON_PNG, dpi=220, bbox_inches="tight"); plt.show()

    # Kết quả từng seed để nhìn trực tiếp các cặp.
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    for ax, metric, title in zip(axes.ravel(), ("profit", "roi", "arr", "sharpe"),
                                 ("Profit", "ROI (%)", "ARR (%)", "Sharpe")):
        pivot = combined.pivot(index="seed", columns="model_key", values=f"test_{metric}")
        for model_key in sorted(expected_models):
            ax.plot(pivot.index, pivot[model_key], marker="o", label=model_key)
        ax.axhline(0.0, color="gray", linestyle="--", linewidth=1)
        ax.set(title=f"Seed-wise Test {title}", xlabel="Seed", ylabel=title)
        ax.grid(alpha=0.25); ax.legend(fontsize=8)
    plt.tight_layout()
    SEEDWISE_PNG = COMPARE_DIR / "ucb009_vs_epsilon_seedwise.png"
    fig.savefig(SEEDWISE_PNG, dpi=220, bbox_inches="tight"); plt.show()

    # Generalization curves nếu output hai máy có episode_curves.
    curve_frames = []
    for path in discover_files("comparison_*_episode_curves.csv"):
        try:
            frame = pd.read_csv(path)
            if {"model_key", "seed", "episode", "split"}.issubset(frame.columns):
                curve_frames.append(frame)
                print("Loaded curves:", path)
        except (OSError, pd.errors.ParserError, pd.errors.EmptyDataError):
            pass
    if curve_frames:
        combined_curves = (pd.concat(curve_frames, ignore_index=True)
                           .drop_duplicates(["model_key", "seed", "episode", "split"], keep="last"))
        fig, axes = plt.subplots(2, 2, figsize=(16, 11))
        for ax, metric, title in zip(axes.ravel(), ("profit", "roi", "arr", "sharpe"),
                                     ("Profit", "ROI (%)", "ARR (%)", "Sharpe")):
            for (model_key, split), group in combined_curves.groupby(["model_key", "split"]):
                mean_curve = group.groupby("episode")[metric].mean()
                style = "-" if split == "train" else "--"
                ax.plot(mean_curve.index, mean_curve.values, linestyle=style,
                        label=f"{model_key} — {split}")
            ax.set(title=f"Generalization Gap — {title}", xlabel="Episode", ylabel=title)
            ax.grid(alpha=0.25); ax.legend(fontsize=7)
        plt.tight_layout()
        CURVE_COMPARE_PNG = COMPARE_DIR / "ucb009_vs_epsilon_generalization.png"
        fig.savefig(CURVE_COMPARE_PNG, dpi=220, bbox_inches="tight"); plt.show()

    print("\n=== FILE SO SÁNH HAI MÔ HÌNH ===")
    for path in [COMPARISON_CSV, PAIRED_CSV, COMPARISON_PNG, SEEDWISE_PNG]:
        print("-", path)